In [0]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
from pyspark.sql import functions as F
from pyspark.sql.functions import col, date_format
import json

CONFIG_PATH = "../config/config.json"
BRONCE_PATH = "abfss://bronze@stdataknowdeveastus001.dfs.core.windows.net/"

In [0]:
dbutils.widgets.dropdown("modo", "automatico", ["historico","automatico"])
dbutils.widgets.text("periodo_final", "")
dbutils.widgets.text("periodo_inicial", "")


modo = dbutils.widgets.get("modo")
periodo_final = dbutils.widgets.get("periodo_final")
periodo_inicial = dbutils.widgets.get("periodo_inicial")   


if modo == "automatico":
  periodo = datetime.now().strftime("%m-%Y")
else :
  periodo = None


In [0]:
with open(CONFIG_PATH) as f:
    cfg = json.load(f)
 
SCOPE       = cfg["key_vault"]["scope"]
KV_KEYS     = cfg["key_vault"]["keys"]

# Leer secretos desde Azure Key Vault (kv-dataknow-dev-eastus)
host     = dbutils.secrets.get(SCOPE, KV_KEYS["host"])
port     = dbutils.secrets.get(SCOPE, KV_KEYS["port"])
db       = dbutils.secrets.get(SCOPE, KV_KEYS["db"])
user     = dbutils.secrets.get(SCOPE, KV_KEYS["user"])
password = dbutils.secrets.get(SCOPE, KV_KEYS["password"])

driver   = "com.microsoft.sqlserver.jdbc.SQLServerDriver"
 
JDBC_URL = (
    f"jdbc:sqlserver://{host}:{port};"
    f"databaseName={db};"
    f"encrypt=true;"
    f"trustServerCertificate=false;"
)
JDBC_PROPS = {
    "user":     user,
    "password": password,
    "driver":   "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

In [0]:
tb_full = [
    "TB_CLIENTES_CORE",
    "TB_PRODUCTOS_CAT",
    "TB_SUCURSALES_RED"
]

tb_incremental = {
    "TB_COMISIONES_LOG" : "fec_cobro",
    "TB_MOV_FINANCIEROS" : "fec_mov",
    "TB_OBLIGACIONES" : "fec_desembolso"
}

In [0]:
def extract_full(list_tb):
    for i in list_tb:
        df = spark.read.format("jdbc") \
            .option("url",      JDBC_URL) \
            .option("dbtable",  "dbo."+i) \
            .option("user",     user) \
            .option("password", password) \
            .option("driver",   driver) \
            .load()

        df = df \
            .withColumn("_fecha_extraccion", F.current_timestamp()) \
            .withColumn("_fuente", F.lit(i))

        df.write.mode("overwrite").format("parquet").save(BRONCE_PATH + i)

extract_full(tb_full)

In [0]:
def generar_periodos(periodo_ini, periodo_fin):
    fmt = "%m-%Y"
    start = datetime.strptime(periodo_ini, fmt)
    end   = datetime.strptime(periodo_fin, fmt)
    periodos = []
    current = start
    while current <= end:
        periodos.append(current.strftime(fmt))
        current += relativedelta(months=1)
    return periodos

def extract_incremental(dict_tb, modo, periodo_ini=None, periodo_fin=None):

    for key, value in dict_tb.items():

        if modo == "automatico":

            periodo = datetime.now().strftime("%m-%Y")

            query = (
                f"(SELECT * FROM dbo.{key} WHERE FORMAT({value}, 'MM-yyyy') = '{periodo}') AS subquery"
            )

            df = (
                spark.read.format("jdbc")
                .option("url", JDBC_URL)
                .option("dbtable", query)
                .option("user", user)
                .option("password", password)
                .option("driver", driver)
                .load()
            )

            if df.limit(1).count() == 0:
                print(f"No hay registros nuevos para {key} | {periodo}")
            else:
                df = (
                    df
                    .withColumn("_fecha_extraccion", F.current_timestamp())
                    .withColumn("_fuente", F.lit(key))
                )
                df.write.mode("overwrite").format("parquet").save(BRONCE_PATH + "/" + key + "/" + periodo)

        elif modo == "historico":

            if not periodo_ini or not periodo_fin:
                raise ValueError("modo historico requiere periodo_inicial y periodo_final en formato MM-yyyy")

            for periodo in generar_periodos(periodo_ini, periodo_fin):
                query = (
                    f"(SELECT * FROM dbo.{key} WHERE FORMAT({value}, 'MM-yyyy') = '{periodo}') AS subquery"
                )
                df = (
                    spark.read.format("jdbc")
                    .option("url", JDBC_URL)
                    .option("dbtable", query)
                    .option("user", user)
                    .option("password", password)
                    .option("driver", driver)
                    .load()
                )
                df = (
                    df
                    .withColumn("_fecha_extraccion", F.current_timestamp())
                    .withColumn("_fuente", F.lit(key))
                )
                df.write.mode("overwrite").format("parquet").save(BRONCE_PATH + "/" + key + "/" + periodo)


extract_incremental(tb_incremental, modo, periodo_inicial, periodo_final)